# CASS — 02. Silhouette Score 분석

`main.py` 실행 후 생성된 탐색 결과(`results/logs/`)를 기반으로
피처 부분집합별 UMAP Silhouette Score를 분석합니다.

**내용:**
1. 탐색 결과 로드
2. Silhouette Score 분포
3. 피처 수 vs Silhouette 관계
4. 피처별 등장 빈도 (상위 부분집합 기준)
5. 최적 부분집합 요약

In [ ]:
import sys
sys.path.insert(0, '..')

import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import LOGS_DIR, FIGURES_DIR

plt.rcParams['figure.dpi'] = 120
print('Import OK')

## 1. 탐색 결과 로드

In [ ]:
# greedy 또는 random 결과 선택
MODE = 'greedy'  # 'greedy' | 'random'
log_path = LOGS_DIR / f'search_results_{MODE}.csv'

df = pd.read_csv(log_path)
# features 컬럼이 문자열로 저장된 경우 파싱
if 'features' in df.columns and isinstance(df['features'].iloc[0], str):
    df['features'] = df['features'].apply(ast.literal_eval)

print(f'결과 수: {len(df)}')
print(df.head(10).to_string())

## 2. Silhouette Score 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['silhouette'], bins=20, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(df['silhouette'].mean(), color='red', linestyle='--', label=f'mean={df["silhouette"].mean():.4f}')
axes[0].set_xlabel('Silhouette Score'); axes[0].set_ylabel('Count')
axes[0].set_title('Silhouette Score 분포', fontweight='bold')
axes[0].legend()

if 'step' in df.columns:
    axes[1].plot(df['step'], df['silhouette'], 'b-o', markersize=5)
    axes[1].set_xlabel('Greedy Step'); axes[1].set_ylabel('Silhouette Score')
    axes[1].set_title('Greedy 탐색 Silhouette 추이', fontweight='bold')
    axes[1].grid(alpha=0.3)
else:
    sc = axes[1].scatter(df['n_features'], df['silhouette'], alpha=0.6, s=40)
    axes[1].set_xlabel('피처 수'); axes[1].set_ylabel('Silhouette Score')
    axes[1].set_title('피처 수 vs Silhouette', fontweight='bold')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'silhouette_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. 피처 수 vs Silhouette (random 모드)

In [ ]:
if 'step' not in df.columns:
    # 피처 수별 Silhouette 박스플롯
    fig, ax = plt.subplots(figsize=(12, 5))
    df.boxplot(column='silhouette', by='n_features', ax=ax)
    ax.set_title('피처 수별 Silhouette Score 분포', fontweight='bold')
    ax.set_xlabel('피처 수'); ax.set_ylabel('Silhouette Score')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

## 4. 상위 부분집합에서 피처 등장 빈도

In [ ]:
# 상위 20% 서브셋에서 피처 등장 빈도
top_n = max(5, len(df) // 5)
top_df = df.nlargest(top_n, 'silhouette')

from collections import Counter
feat_counter = Counter()
for feats in top_df['features']:
    feat_counter.update(feats)

freq_df = pd.DataFrame(feat_counter.most_common(), columns=['Feature', 'Count'])
freq_df['Ratio'] = freq_df['Count'] / top_n

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(freq_df['Feature'][::-1], freq_df['Ratio'][::-1], color='teal', alpha=0.8)
ax.set_xlabel('상위 서브셋 내 등장 비율')
ax.set_title(f'상위 {top_n}개 부분집합 내 피처 빈도 (Silhouette 기준)', fontweight='bold')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.6, label='50% 기준선')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_frequency_top_subsets.png', dpi=200, bbox_inches='tight')
plt.show()

print(freq_df.to_string(index=False))

## 5. 최적 부분집합 요약

In [ ]:
best_row = df.loc[df['silhouette'].idxmax()]
print('=== 최적 부분집합 (Silhouette 최고) ===')
print(f'  Silhouette Score : {best_row["silhouette"]:.4f}')
print(f'  피처 수          : {best_row["n_features"]}')
print(f'  피처 목록:')
for f in best_row['features']:
    print(f'    - {f}')